# Step 13 — Multi-Agent

🇬🇧 **English** (this notebook)

Add a second agent with a different, complementary role, and wire both into a `Crew`. You'll run the same two agents under two different `Process` strategies: `sequential` — the same fixed pipeline this repo's full demo project (`src/research_crew/crew.py`) uses, just defined directly in this notebook instead of via `@CrewBase` and YAML config files — and `hierarchical`, where a manager decides at runtime which agent handles each task, instead of that being fixed in code.

## Learning objective

By the end of this notebook, you will:

- Understand why a second, complementary agent can produce something neither agent produces alone
- Have chained two `Task`s together with `context=[...]`, so one agent's output feeds directly into another's input
- Have run your first real `Crew` with two agents and `process=Process.sequential`
- Understand how `process=Process.hierarchical` differs: task order is still fixed, but *which* agent handles each task is a manager's runtime decision, not something wired into the code
- Have run the same two agents under a manager, via `manager_llm` (or a custom `manager_agent`)

## Prerequisites

- [Step 09 — Single Agent](step_09_single_agent.ipynb) completed — this notebook reuses its Researcher agent unchanged
- The same `.env` setup as the previous steps

## Background

A single agent can do multiple things, but it can't hold two genuinely different epistemic roles at the same time — it can't be simultaneously credulous (collecting everything) and skeptical (questioning what it found). Two agents let you encode that tension into the architecture. The seminal demonstration of agents collaborating through conversation was:

> Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)

## How this works

Two agents and two tasks, wired into a `Crew` with `process=Process.sequential`:

1. **`researcher`** is Step 09's exact same Researcher agent, unchanged.
2. **`analyst`** is a new agent with a different `role`/`goal`/`backstory` — someone who turns raw findings into a report for a specific audience, a job the Researcher was never asked to do.
3. **`research_task`** is assigned to the Researcher; **`analysis_task`** is assigned to the Analyst, with one crucial addition: `context=[research_task]`. This tells CrewAI to feed the first task's output into the second automatically — the same idea as Step 06's chain prompting, but wired declaratively instead of copy-pasted by hand.
4. **`crew.kickoff()`** runs both tasks in the order they appear in `tasks=[...]`, because `process=Process.sequential`.

Everything else in the cell (the `tracing=True` flag and the `crewai_event_bus.flush()` call afterward) is optional bookkeeping that prints a shareable trace link — it doesn't affect what the agents do.

Further down, the same two agents run again under `process=Process.hierarchical` — a different way of wiring the same pieces together.

In [1]:
import os

from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process

load_dotenv()

topic = "EU AI Act"

# ── Agent 1: Researcher — same "researcher" template as agents.yaml ──────────
researcher = Agent(
    role=f"{topic} Senior Data Researcher",
    goal=f"Uncover cutting-edge developments in {topic}",
    backstory=(
        f"You're a seasoned researcher with a knack for uncovering the latest "
        f"developments in {topic}. Known for your ability to find the most relevant "
        f"information and present it in a clear and concise manner."
    ),
    verbose=True,
)

# ── Agent 2: Analyst — same "analyst" template as agents.yaml ─────────────────
analyst = Agent(
    role=f"{topic} Reporting Analyst",
    goal=f"Create detailed reports based on {topic} data analysis and research findings",
    backstory=(
        "You're a meticulous analyst with a keen eye for detail. You're known for "
        "your ability to turn complex data into clear and concise reports, making "
        "it easy for others to understand and act on the information you provide."
    ),
    verbose=True,
)

# ── Task 1: research — assigned to the Researcher ─────────────────────────────
research_task = Task(
    description=(
        f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
        "limited, minimal) and what obligations apply to providers of high-risk AI systems."
    ),
    expected_output=f"A structured summary of {topic}'s risk categories and obligations.",
    agent=researcher,
)

# ── Task 2: analysis — assigned to the Analyst, chained to Task 1 via `context` ─
analysis_task = Task(
    description=(
        "Using the research findings, write a short report for a product team that ships "
        "an AI-powered hiring tool in the EU. Call out exactly which obligations apply "
        "to them and by when."
    ),
    expected_output="A short, actionable report for a product team, grounded in the research findings.",
    agent=analyst,
    context=[research_task],
)

# ── Crew — same sequential process as src/research_crew/crew.py ──────────────
crew = Crew(
    agents=[researcher, analyst],
    tasks=[research_task, analysis_task],
    process=Process.sequential,
    verbose=True,
    # Prints a free, no-signup shareable trace URL (agent reasoning, task
    # timing, tool calls) to app.crewai.com after this cell finishes.
    tracing=True,
)

# ── Process — kick off the crew ───────────────────────────────────────────────
result = crew.kickoff()

# Trace upload happens on a background thread; a plain script naturally waits
# for it at process exit, but a notebook's kernel keeps running - flush() blocks
# until it's done, so the trace link below is guaranteed to print in this cell.
from crewai.events import crewai_event_bus
crewai_event_bus.flush()

print("=== Researcher output ===")
print(research_task.output.raw)
print("\n=== Analyst output ===")
print(analysis_task.output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  2cf60bf8-1215-45f4-b1fe-341cd199e40b                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│  ID: 29abe296-4070-45ae-bf20-33df9258135d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Task: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  As a Senior Data Researcher specializing in the EU AI Act, I provide the following breakdown of the            │
│  risk-based framework. The Act employs a tiered approach, calibrating regulatory burden according to the        │
│  potential harm an AI system poses to the safety, fundamental rights, and democratic values of EU citizens.     │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Part I: The Four Risk-Based Categories                                                                     │
│                                                                                                                 │
│  The EU AI Act categorizes AI systems based on their intended purpose and risk profile:                         │
│                                                                                                                 │
│  1.  **Unacceptable Risk (Prohibited):** Systems deemed a clear threat to fundamental rights.                   │
│      *   *Examples:* Social scoring systems by governments; real-time biometric identification in public        │
│  spaces by law enforcement (with narrow exceptions); AI that uses subliminal techniques to distort behavior;    │
│  AI exploiting vulnerabilities of specific groups (age, disability).                                            │
│  2.  **High-Risk:** Systems that pose significant risks to health, safety, or fundamental rights.               │
│      *   *Examples:* AI used in critical infrastructure, education/vocational training,                         │
│  employment/recruitment, essential private/public services (e.g., credit scoring), and certain law enforcement  │
│  or border control applications.                                                                                │
│  3.  **Limited Risk (Transparency Obligations):** Systems that require specific disclosures to ensure users     │
│  know they are interacting with AI.                                                                             │
│      *   *Examples:* Chatbots (AI-generated text), emotion recognition systems (if not high-risk), and          │
│  deepfakes (AI-generated or manipulated content).                                                               │
│  4.  **Minimal Risk (No Regulation):** The vast majority of AI systems currently in the EU market.              │
│      *   *Examples:* Spam filters, AI-enabled video games, or inventory management software. These are subject  │
│  only to existing safety legislation and voluntary codes of conduct.                                            │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ### Part II: Obligations for Providers of High-Risk AI Systems                                                 │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what obligations     │
│  apply to providers of high-risk AI systems.                                                                    │
│  Agent:                                                                                                         │
│  EU AI Act Senior Data Researcher                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Reporting Analyst                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **To:** AI Product Team                                                                                        │
│  **From:** EU AI Act Reporting Analyst                                                                          │
│  **Subject:** Compliance Requirements for AI-Powered Hiring Tools                                               │
│                                                                                                                 │
│  ### Executive Summary                                                                                          │
│  Under the EU AI Act, AI systems used for **recruitment, selection, and employment** (including CV screening,   │
│  ranking candidates, and analyzing behavioral/personality traits) are strictly classified as **High-Risk AI**.  │
│  This classification is mandated because these systems directly impact fundamental rights and access to         │
│  employment.                                                                                                    │
│                                                                                                                 │
│  ### Mandatory Obligations                                                                                      │
│  As providers of a high-risk AI system, our team must implement the following operational and technical         │
│  requirements to remain compliant with EU law:                                                                  │
│                                                                                                                 │
│  1.  **Risk Management System:** Establish a lifecycle-wide system to identify and mitigate risks related to    │
│  fairness, bias, and performance.                                                                               │
│  2.  **Data Governance:** Implement strict protocols for training and validation datasets to ensure they are    │
│  relevant, representative, and free of biases that could cause discriminatory hiring outcomes.                  │
│  3.  **Technical Documentation:** Maintain comprehensive documentation demonstrating compliance, which must be  │
│  accessible to national authorities upon request.                                                               │
│  4.  **Transparency:** Design the tool to be transparent; provide clear instructions for HR users, including    │
│  defined system limitations and mandatory human oversight procedures.                                           │
│  5.  **Human Oversight:** Ensure the tool is built so a human operator can effectively monitor, override, or    │
│  reverse automated decisions.                                                                                   │
│  6.  **Robustness & Cybersecurity:** Ensure the system is resilient against unauthorized manipulation and       │
│  maintains consistent performance (accuracy) throughout its lifecycle.                                          │
│  7.  **Quality Management System (QMS):** Maintain an internal QMS that includes automatic event logging (to    │
│  track system behavior) and a robust post-market monitoring process.                                            │
│  8.  **Conformity Assessment & CE Marking:** Perform a formal conformity assessment to verify compliance. Once  │
│  compliant, the tool must bear the **CE marking** befor

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the research findings, write a short report for a product team that ships an AI-powered hiring tool in   │
│  the EU. Call out exactly which obligations apply to them and by when.                                          │
│  Agent:                                                                                                         │
│  EU AI Act Reporting Analyst                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  2cf60bf8-1215-45f4-b1fe-341cd199e40b                                                                           │
│  Final Output: **To:** AI Product Team                                                                          │
│  **From:** EU AI Act Reporting Analyst                                                                          │
│  **Subject:** Compliance Requirements for AI-Powered Hiring Tools                                               │
│                                                                                                                 │
│  ### Executive Summary                                                                                          │
│  Under the EU AI Act, AI systems used for **recruitment, selection, and employment** (including CV screening,   │
│  ranking candidates, and analyzing behavioral/personality traits) are strictly classified as **High-Risk AI**.  │
│  This classification is mandated because these systems directly impact fundamental rights and access to         │
│  employment.                                                                                                    │
│                                                                                                                 │
│  ### Mandatory Obligations                                                                                      │
│  As providers of a high-risk AI system, our team must implement the following operational and technical         │
│  requirements to remain compliant with EU law:                                                                  │
│                                                                                                                 │
│  1.  **Risk Management System:** Establish a lifecycle-wide system to identify and mitigate risks related to    │
│  fairness, bias, and performance.                                                                               │
│  2.  **Data Governance:** Implement strict protocols for training and validation datasets to ensure they are    │
│  relevant, representative, and free of biases that could cause discriminatory hiring outcomes.                  │
│  3.  **Technical Documentation:** Maintain comprehensive documentation demonstrating compliance, which must be  │
│  accessible to national authorities upon request.                                                               │
│  4.  **Transparency:** Design the tool to be transparent; provide clear instructions for HR users, including    │
│  defined system limitations and mandatory human oversight procedures.                                           │
│  5.  **Human Oversight:** Ensure the tool is built so a human operator can effectively monitor, override, or    │
│  reverse automated decisions.                                                                                   │
│  6.  **Robustness & Cybersecurity:** Ensure the system is resilient against unauthorized manipulation and       │
│  maintains consistent performance (accuracy) throughout its lifecycle.                                          │
│  7.  **Quality Management System (QMS):** Maintain an internal QMS that includes automatic event logging (to    │
│  track system behavior) and a robust post-market monit

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: 8d39083d-d1c2-4df0-af43-a0b0c0ba4f45                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/8d39083d-d1c2-4df0-af43-a0b0c0ba4f45?access_code=TRA │
│ CE-6ebb1b8ee7                                                                                                   │
│ 🔑 Access Code: TRACE-6ebb1b8ee7                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

=== Researcher output ===
As a Senior Data Researcher specializing in the EU AI Act, I provide the following breakdown of the risk-based framework. The Act employs a tiered approach, calibrating regulatory burden according to the potential harm an AI system poses to the safety, fundamental rights, and democratic values of EU citizens.

---

### Part I: The Four Risk-Based Categories

The EU AI Act categorizes AI systems based on their intended purpose and risk profile:

1.  **Unacceptable Risk (Prohibited):** Systems deemed a clear threat to fundamental rights.
    *   *Examples:* Social scoring systems by governments; real-time biometric identification in public spaces by law enforcement (with narrow exceptions); AI that uses subliminal techniques to distort behavior; AI exploiting vulnerabilities of specific groups (age, disability).
2.  **High-Risk:** Systems that pose significant risks to health, safety, or fundamental rights.
    *   *Examples:* AI used in critical infrastructure,

### A non-sequential pattern: manager delegation (`Process.hierarchical`)

Above, each `Task` is hard-wired to one `Agent` via `agent=...`, and `crew.kickoff()` runs them in the order they appear in `tasks=[...]`. `Process.hierarchical` removes that fixed wiring: the tasks below don't get an `agent=` at all. Instead, a **manager** — either an `Agent` CrewAI builds for you from `manager_llm`, or your own `Agent` passed as `manager_agent` — reads each task and decides, at that moment, which of the crew's agents is best suited, then delegates to them via a tool call.

Two things worth being precise about, since "hierarchical" invites the wrong mental model:

- **Task order still isn't dynamic.** The `for` loop over `tasks=[...]` doesn't change — task 1 still runs before task 2. What's dynamic is *who* does each task, decided fresh every run, not *when* it happens.
- **The manager is not one of your agents.** It's a separate role that does no task work itself — it only reads tasks and delegates. CrewAI enforces this: a `manager_agent` may not also appear in `agents=[...]`, and it may not be given `tools=` (only the agents it delegates to can have tools).

The `researcher` and `analyst` from above are reused unchanged below — same `role`/`goal`/`backstory`, no code changes to either agent. Only the `Crew`'s wiring is different.

In [2]:
# ── Same two tasks, but neither gets an `agent=` — the manager decides who does what ──
research_task_h = Task(
    description=(
        f"Explain {topic}'s risk-based categories (unacceptable, high-risk, "
        "limited, minimal) and what obligations apply to providers of high-risk AI systems."
    ),
    expected_output=f"A structured summary of {topic}'s risk categories and obligations.",
)

analysis_task_h = Task(
    description=(
        "Using the research findings, write a short report for a product team that ships "
        "an AI-powered hiring tool in the EU. Call out exactly which obligations apply "
        "to them and by when."
    ),
    expected_output="A short, actionable report for a product team, grounded in the research findings.",
)

# ── Crew — same two agents as coworkers, orchestrated by a manager instead of fixed code ──
hierarchical_crew = Crew(
    agents=[researcher, analyst],  # the manager's coworkers — the manager itself is not one of them
    tasks=[research_task_h, analysis_task_h],
    process=Process.hierarchical,
    manager_llm=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"),
    verbose=True,
)

# ── Process — kick off the crew ───────────────────────────────────────────────
result_h = hierarchical_crew.kickoff()

print("=== Research task, as delegated by the manager ===")
print(research_task_h.output.raw)
print("\n=== Analysis task, as delegated by the manager ===")
print(analysis_task_h.output.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  0a12752d-0a8b-429a-85de-9c27f91d10b4                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│  ID: 1df9d9cf-81ef-46f5-9576-cdbc22b417a7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Explain EU AI Act's risk-based categories (unacceptable, high-risk, limited, minimal) and what           │
│  obligations apply to providers of high-risk AI systems.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: delegate_work_to_coworker                                                                                │
│  Args: {'coworker': 'EU AI Act Senior Data Researcher', 'task': "Provide a detailed explanation of the EU AI    │
│  Act's four risk categories (Unacceptable, High-Risk, Limited, Minimal) and outline the specific ob...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: EU AI Act Senior Data Researcher                                                                        │
│                                                                                                                 │
│  Task: Provide a detailed explanation of the EU AI Act's four risk categories (Unacceptable, High-Risk,         │
│  Limited, Minimal) and outline the specific obligations for providers of high-risk AI systems. Ensure the       │
│  response is structured as a clear, comprehensive summary.                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Internal Memo: EU AI Act Compliance Strategy for AI-Powered Hiring Tools                                   │
│                                                                                                                 │
│  **To:** Product Team                                                                                           │
│  **Subject:** Mandatory Compliance Requirements under the EU AI Act                                             │
│                                                                                                                 │
│  As we operate an AI-powered hiring tool in the European Union, our system is classified as **"High-Risk"**     │
│  under Annex III of the EU AI Act. Because our tool directly impacts employment, recruitment, and worker        │
│  management, we are subject to stringent regulatory obligations.                                                │
│                                                                                                                 │
│  #### 1. Compliance Deadline                                                                                    │
│  *   **Full Compliance Date:** We must be in full compliance with all high-risk requirements by **August 2,     │
│  2027**.                                                                                                        │
│  *   **Strategic Milestone:** We should aim to have our core Quality Management System (QMS) and risk           │
│  mitigation frameworks fully operational by **August 2, 2026**, to avoid last-minute certification              │
│  bottlenecks.                                                                                                   │
│  *   **Important Note on "Significant Changes":** If we implement a major architectural overhaul or change the  │
│  intended purpose of our hiring tool before August 2027, the system may be treated as a "new" system,           │
│  triggering immediate compliance obligations.                                                                   │
│                                                                                                                 │
│  #### 2. Mandatory Obligations for Our Hiring Tool                                                              │
│  To ensure we can continue to operate in the EU market, our product development and operations must prioritize  │
│  the following:                                                                                                 │
│                                                                                                                 │
│  *   **Risk Management System:** Establish an iterative, lifecycle-wide process to identify, estimate, and      │
│  mitigate risks to fundamental rights (e.g., bias in candidate screening).                                      │
│  *   **Data Governance:** Ensure training and validation datasets are representative, error-free, and           │
│  rigorously tested for bias to prevent discriminatory hiring outcomes.                                          │
│  *   **Technical Documentation:** Maintain a comprehensive file demonstrating compliance, including system      │
│  design, logic, and performance metrics, to be readily available for regulatory review.                         │
│  *   **Record-Keeping (Logging):** Implement automatic 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Using the research findings, write a short report for a product team that ships an AI-powered hiring tool in   │
│  the EU. Call out exactly which obligations apply to them and by when.                                          │
│  Agent:                                                                                                         │
│  Crew Manager                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  0a12752d-0a8b-429a-85de-9c27f91d10b4                                                                           │
│  Final Output: ### Internal Memo: EU AI Act Compliance Strategy for AI-Powered Hiring Tools                     │
│                                                                                                                 │
│  **To:** Product Team                                                                                           │
│  **Subject:** Mandatory Compliance Requirements under the EU AI Act                                             │
│                                                                                                                 │
│  As we operate an AI-powered hiring tool in the European Union, our system is classified as **"High-Risk"**     │
│  under Annex III of the EU AI Act. Because our tool directly impacts employment, recruitment, and worker        │
│  management, we are subject to stringent regulatory obligations.                                                │
│                                                                                                                 │
│  #### 1. Compliance Deadline                                                                                    │
│  *   **Full Compliance Date:** We must be in full compliance with all high-risk requirements by **August 2,     │
│  2027**.                                                                                                        │
│  *   **Strategic Milestone:** We should aim to have our core Quality Management System (QMS) and risk           │
│  mitigation frameworks fully operational by **August 2, 2026**, to avoid last-minute certification              │
│  bottlenecks.                                                                                                   │
│  *   **Important Note on "Significant Changes":** If we implement a major architectural overhaul or change the  │
│  intended purpose of our hiring tool before August 2027, the system may be treated as a "new" system,           │
│  triggering immediate compliance obligations.                                                                   │
│                                                                                                                 │
│  #### 2. Mandatory Obligations for Our Hiring Tool                                                              │
│  To ensure we can continue to operate in the EU market, our product development and operations must prioritize  │
│  the following:                                                                                                 │
│                                                                                                                 │
│  *   **Risk Management System:** Establish an iterative, lifecycle-wide process to identify, estimate, and      │
│  mitigate risks to fundamental rights (e.g., bias in candidate screening).                                      │
│  *   **Data Governance:** Ensure training and validation datasets are representative, error-free, and           │
│  rigorously tested for bias to prevent discriminatory hiring outcomes.                                          │
│  *   **Technical Documentation:** Maintain a comprehen

=== Research task, as delegated by the manager ===
### The EU AI Act: Risk-Based Framework and Obligations

The EU AI Act adopts a risk-based approach, categorizing AI systems based on the potential impact they pose to the health, safety, and fundamental rights of EU citizens. Obligations scale proportionally with the level of risk identified.

#### I. Risk Categories

1. **Unacceptable Risk (Prohibited AI):**
   Systems deemed to contravene European values are strictly prohibited. These include:
   * **Cognitive Behavioral Manipulation:** Techniques designed to distort behavior in a way that causes physical or psychological harm.
   * **Exploitation:** Targeting vulnerable groups (due to age or disability) to alter their behavior.
   * **Social Scoring:** Public authorities classifying individuals based on behavior or personality traits.
   * **Biometric Surveillance:** Real-time remote biometric identification in public spaces for law enforcement (with very narrow exceptions).
   * *

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Your task

1. Run the sequential cell. Compare the Analyst's report to Step 09's Researcher answer alone — is it more targeted and actionable, or does it mostly repackage the same content?

2. **Experiment**: remove `context=[research_task]` from `analysis_task` and rerun. Without that link, does the Analyst still somehow reference the Researcher's specific findings, or does it write a generic report from scratch?

3. Run the hierarchical cell. With `verbose=True`, the console output shows the manager's own reasoning and its "Delegate work to coworker" / "Ask question to coworker" tool calls — find the line where it picks the Researcher for the first task and the Analyst for the second. Compare `result_h` to `result` from the sequential run: same two jobs, same two agents — did the manager's routing produce a meaningfully different outcome, or basically the same one with extra steps (and extra LLM calls) in between?

4. Now swap in your own team's topic — keep the Researcher agent from Step 09, and give it a second, genuinely complementary role and task suited to your topic.

5. This is where your project shifts from guided exercises to your own design: use everything from Steps 03–13 to build and document your own agent. Fill in `REPORT.md` — see [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full report structure and what's expected. Make sure the **Sprint Progression** table has all five rows filled in — it's the fastest way for your reader to see the arc before they read the rest of the report.

## Shortcomings

The sequential pipeline is fixed: the Researcher always runs first, the Analyst always runs second, and the order never changes based on what either agent actually finds. `Process.hierarchical` fixes *who* handles each task, not *when* — the task order in `tasks=[...]` is still fixed, and it costs more: every task now takes an extra LLM call (the manager's own reasoning) on top of whichever agent it delegates to, and the manager's choice of delegate isn't guaranteed to be consistent between runs. For two agents with clearly non-overlapping roles like these, that extra machinery mostly buys you nothing — it starts to pay off once you have enough agents, or similar-enough roles, that hard-coding `agent=...` on every task stops being obvious.

Neither `Crew` above uses any of the tools/MCP/RAG grounding from Steps 10–12 — combining multi-agent orchestration with a grounded, tool-using Researcher is a natural next experiment, just not one this notebook walks you through.

This is the last step in the exercise series. From here, the main [README's use-case table](../../README.md#use-cases-to-pick-from) and `REPORT.md` ask you to step back and judge, for your own topic, which combination of everything covered — single vs. multi-agent, sequential vs. hierarchical, tools, MCP, RAG — is actually worth the added complexity.

## Resources for further reading

- Wu, Q., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation*. [arXiv:2308.08155](https://arxiv.org/abs/2308.08155)
- [CrewAI `Process` concept docs](https://docs.crewai.com/en/concepts/processes) — `sequential` vs. `hierarchical`, covered briefly in Step 02

## Stretch goal

Add a third agent whose absence you'd actually notice — a critic, a translator for a non-expert audience, or a validator. The hierarchical crew is the more natural fit for this: add the agent to `agents=[...]` and a `Task` for it to `tasks=[...]`, but don't pin it with `agent=...` — let the manager decide on its own whether and when to bring it in, rather than wiring it into a fixed spot in the pipeline like you'd have to for the sequential version. Does the output meaningfully change? If it doesn't, ask yourself why.

---

**This is your final submission.** See [Assignment Overview](../../team_assignment/en/assignment-overview.md) for the full grading rubric and exactly what to submit.